<a href="https://colab.research.google.com/github/miraitech24/public/blob/main/%E6%99%82%E8%B6%85%E3%81%88%E3%82%B1%E3%83%B3%E3%82%BF/kenta_4choice_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 🌌 時超えケンタ (CHRONO KENTA): 4択SFアドベンチャー (全話＆番外編対応完全版)
# ==============================================================================
# 【特徴】
# ・GitHub(miraitech24/public) の全画像を自動取得＆スマート画像マッチング！
# ・存在しないページ名(page1等)が指定されても同エピソードの実在画像(page2等)を自動代替表示！
# ・GitHub Raw URLからの直接画像読み込みにも完全対応！
# ・選択肢ボタンの並び順は毎回ランダム（位置推測不可能）！
# ==============================================================================

import os
import json
import glob
import random
import zipfile
import urllib.request
import urllib.parse
import subprocess
from IPython.display import Image, display, clear_output
import ipywidgets as widgets

# ------------------------------------------------------------------------------
# 1. 画像検索・GitHub自動同期＆スマート代替マッチングユーティリティ
# ------------------------------------------------------------------------------
def sync_github_repo():
    """Colab起動時に GitHub(miraitech24/public) から最新アセットを取得"""
    try:
        if not os.path.exists("public"):
            print("🌐 GitHub (miraitech24/public) より最新アセットを同期中...")
            subprocess.run(["git", "clone", "https://github.com/miraitech24/public.git"], check=False)

        # zipファイルがあれば解凍
        zip_matches = glob.glob("/content/**/*.zip", recursive=True) + glob.glob("./**/*.zip", recursive=True)
        for zf in set(zip_matches):
            try:
                target_dir = "/content/sample_data"
                os.makedirs(target_dir, exist_ok=True)
                with zipfile.ZipFile(zf, 'r') as zip_ref:
                    zip_ref.extractall(target_dir)
            except Exception:
                pass
    except Exception as e:
        print("⚠️ 自動同期注意:", e)

sync_github_repo()

def find_image_file(img_name):
    """画像のファイル名から再帰検索＆スマート代替マッチング"""
    if not img_name:
        return None, False

    filename_only = os.path.basename(img_name)

    # 1. ローカルディスクからの直接検索
    search_patterns = [
        f"/content/**/{filename_only}",
        f"./**/{filename_only}",
        f"/content/sample_data/**/{filename_only}"
    ]
    for pattern in search_patterns:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0], False

    # 2. ページ番号不一致の場合のスマートフォールバック (例: page1がない場合は ep1 の最初の画像を使用)
    ep_match = None
    if "ep1" in filename_only or "prologue" in filename_only:
        ep_match = "ep1"
    elif "ep2" in filename_only:
        ep_match = "ep2"
    elif "ep3" in filename_only:
        ep_match = "ep3"
    elif "ep4" in filename_only:
        ep_match = "ep4"
    elif "epilogue" in filename_only:
        ep_match = "epilogue"
    elif "sp2" in filename_only:
        ep_match = "sp2"
    elif "sp3" in filename_only:
        ep_match = "sp3"
    elif "sp4" in filename_only:
        ep_match = "sp4"
    elif "sp5" in filename_only:
        ep_match = "sp5"

    if ep_match:
        fallback_matches = glob.glob(f"/content/**/*kenta_{ep_match}_*.png", recursive=True) + glob.glob(f"./**/*kenta_{ep_match}_*.png", recursive=True)
        if fallback_matches:
            fallback_matches.sort()
            return fallback_matches[0], False

    # 3. GitHub Raw URL からの直接表示フォールバック
    encoded_folder = urllib.parse.quote("時超えケンタ")
    encoded_file = urllib.parse.quote(filename_only)
    raw_url = f"https://raw.githubusercontent.com/miraitech24/public/main/{encoded_folder}/{encoded_file}"
    return raw_url, True

# ------------------------------------------------------------------------------
# 2. シナリオデータベース (実在する画像ファイル名に完全適合)
# ------------------------------------------------------------------------------
EMBEDDED_DB = {
  "game_title": "時超えケンタ (CHRONO KENTA): 星間ハビタブルゾーン同盟",
  "initial_state": {
    "velocity_c": 0.20,
    "hull_pct": 100.0,
    "fuel_tons": 100.0,
    "crawlers_count": 10,
    "delay_years": 4.2
  },
  "scenes": {
    "scene_prologue": {
      "chapter_name": "プロローグ: 人類文明の黄昏と水星点火決断",
      "description": "西暦2070年代。地球環境の悪化と太陽系資源の枯渇に直面した人類は、4.2光年先のプロキシマ・ケンタウリ系へ物理AI『時超えケンタ』を派遣する決断を下す。",
      "stage_id": "prologue",
      "image_file": "kenta_ep1_page2_manga.png",
      "options": [
        {
          "label": "水星ダイソン環の全出力を集光し、光速20%への加速レーザーを点火する",
          "delta": {"velocity_c": 0.20},
          "result_text": "【物理的帰結】1.64PWのレーザーが深宇宙へ放射され、ケンタ号は0.2cへと加速を開始した。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        },
        {
          "label": "安全係数を優先し、出力を10%に抑えて超低速・長期航海を選ぶ",
          "delta": {"velocity_c": 0.02, "delay_years": 40.0},
          "result_text": "【物理的帰結】船体負荷は最小で済んだが、到達まで200年以上を要する超長航海となった。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        },
        {
          "label": "光圧加速を取りやめ、化学・核融合エンジンのみで発進する",
          "delta": {"fuel_tons": -90.0, "velocity_c": 0.01},
          "result_text": "【物理的帰結】出航直後に大量の燃料を喪失。星間加速に必要なエネルギーが枯渇した。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        },
        {
          "label": "恒星レーザーの照準テストを行わずにいきなりフルパワー点火する",
          "delta": {"hull_pct": -80.0},
          "result_text": "【物理的帰結】照射ブレが発生！船首セイルの偏心熱膨張により、初期加速段階で大ダメージを受けた。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        }
      ]
    },
    "scene_ep1": {
      "chapter_name": "第1話「種火」: 水星ダイソン環（1.64PW）からの光圧点火",
      "description": "水星ダイソン環からの1.64PWレーザーを受け、超巨大ミラーセイルを展開して1年間加速。光速の20%（0.2c）へ到達した。セイルの熱負荷限界が迫る！",
      "stage_id": "ep1",
      "image_file": "kenta_ep1_page2_manga.png",
      "options": [
        {
          "label": "ミラーセイルを予定通り切り離し、0.2cで21年間の巡航へ移行",
          "delta": {"velocity_c": 0.0, "hull_pct": 0.0},
          "result_text": "【物理的帰結】セイルの熱放射限界前に正常パージ完了。船体は慣性航行に入った。",
          "next_scene": "scene_ep2",
          "is_gameover": False
        },
        {
          "label": "追加加速を狙い、セイル限界を超えてレーザー照射を継続する",
          "delta": {"hull_pct": -40.0, "velocity_c": 0.03},
          "result_text": "【物理的帰結】0.23cへ微増したものの、熱膨張により船首センサー群が半壊した。",
          "next_scene": "scene_ep2",
          "is_gameover": False
        },
        {
          "label": "セイルの角度を傾けて太陽系外縁のダスト帯を回避しながら進む",
          "delta": {"velocity_c": -0.01},
          "result_text": "【物理的帰結】安全な航路を確保したが、わずかに減速した。",
          "next_scene": "scene_ep2",
          "is_gameover": False
        },
        {
          "label": "セイルを切り離さずに船体に巻き付け、追加の磁気シールドとして利用する",
          "delta": {"hull_pct": -90.0},
          "result_text": "【物理的帰結】高熱を持ったセイルが船体装甲を融解！致命的な損傷を負った。",
          "next_scene": "scene_ep2",
          "is_gameover": True
        }
      ]
    },
    "scene_ep2": {
      "chapter_name": "第2話「21年の旅」: 減速減圧と電磁シールド展開",
      "description": "プロキシマ・ケンタウリ系に接近。秒速6万キロの超高速で進入するため、プラズマ減速と電磁シールドを最大展開して星間物質の衝突に備える！",
      "stage_id": "ep2",
      "image_file": "kenta_ep2_page1_manga.png",
      "options": [
        {
          "label": "前方に超電導磁気シールドを展開し、星間プラズマを減速ブレーキとして利用する",
          "delta": {"velocity_c": -0.15, "hull_pct": -5.0},
          "result_text": "【物理的帰結】磁気ブレーキングが成功！安全な進入速度まで減速完了。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        },
        {
          "label": "減速を行わず、そのまま最接近して周回軌道への逆噴射一発にかける",
          "delta": {"fuel_tons": -80.0},
          "result_text": "【物理的帰結】大量の燃料を消耗したが、逆噴射により何とか減速に成功した。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        },
        {
          "label": "プロキシマ・ケンタウリの強磁場を利用してスイングバイ減速を行う",
          "delta": {"delay_years": 0.5},
          "result_text": "【物理的帰結】恒星のフレア放射を浴びて通信遅延が拡大したが、軌道投入に成功。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        },
        {
          "label": "シールド展開を省略し、船体装甲のみで星間ガス群を突き抜ける",
          "delta": {"hull_pct": -100.0},
          "result_text": "【物理的帰結】微細な星間物質が秒速6万キロで船体を穿ち、ケンタ号は蒸発した。",
          "next_scene": "scene_ep3",
          "is_gameover": True
        }
      ]
    },
    "scene_ep3": {
      "chapter_name": "第3話「赤き大地」: プロキシ b 着陸と着水・プラント建設",
      "description": "潮汐固定されたプロキシマb。潮汐境界領域（終末帯）に着陸。強烈な放射線と猛風の中、クローラー群を起動して最初のプラントを建設する！",
      "stage_id": "ep3",
      "image_file": "kenta_ep3_page3_manga.png",
      "options": [
        {
          "label": "終末帯の地下洞窟（溶岩チューブ）に自律クローラーを投下し拠点を設営する",
          "delta": {"crawlers_count": 0, "hull_pct": 5.0},
          "result_text": "【物理的帰結】放射線を100%遮断できる天然の地下要塞拠点が完成した。",
          "next_scene": "scene_ep4",
          "is_gameover": False
        },
        {
          "label": "昼半球の太陽光全域に大型ソーラープラントを一気に広げて建設する",
          "delta": {"crawlers_count": -3, "fuel_tons": 20.0},
          "result_text": "【物理的帰結】恒星フレアの熱負荷でクローラー数台が故障したが、膨大なパワーを確保。",
          "next_scene": "scene_ep4",
          "is_gameover": False
        },
        {
          "label": "夜半球の氷床エリアへ向かい、熱源プラントで氷を溶かして水資源を確保する",
          "delta": {"fuel_tons": 50.0},
          "result_text": "【物理的帰結】大規模な水・水素生成プラントの稼働を開始した。",
          "next_scene": "scene_ep4",
          "is_gameover": False
        },
        {
          "label": "防護策を講じず、平原の地上露天にコアプラントを露出設置する",
          "delta": {"crawlers_count": -10},
          "result_text": "【物理的帰結】スーパーフレアの直撃を受け、すべての地上設備が焼き払われた。",
          "next_scene": "scene_ep4",
          "is_gameover": True
        }
      ]
    },
    "scene_ep4": {
      "chapter_name": "第4話「歪む時空」: マイクロワームホール(10^-15m)開通",
      "description": "地球とのリアルタイム通信と物質転送のため、負のエネルギー密度（エキゾチック物質）を用いて極微ワームホールを開通させる！",
      "stage_id": "ep4",
      "image_file": "kenta_ep4_page3_manga.png",
      "options": [
        {
          "label": "エキゾチック物質を安定注入し、10^-15mのマイクロワームホール口を固定する",
          "delta": {"delay_years": -4.2},
          "result_text": "【物理的帰結】地球との量子もつれ情報通信が開通！通信遅延がゼロになった。",
          "next_scene": "scene_sp2",
          "is_gameover": False
        },
        {
          "label": "一気に喉（スロート）を人間が通過できるマクロサイズまで急速拡張する",
          "delta": {"hull_pct": -50.0},
          "result_text": "【物理的帰結】時空の歪みによる潮汐剪断力が発生！施設の一部が破壊された。",
          "next_scene": "scene_sp2",
          "is_gameover": False
        },
        {
          "label": "安全のため、ワームホール出力を最小限に絞りバックアップ回線のみ確立する",
          "delta": {"delay_years": -2.0},
          "result_text": "【物理的帰結】部分的な通信速度改善にとどまったが、完全に安全運用が達成された。",
          "next_scene": "scene_sp2",
          "is_gameover": False
        },
        {
          "label": "負のエネルギー供給のフィードバック制御を切って連続運用する",
          "delta": {"hull_pct": -100.0},
          "result_text": "【物理的帰結】ワームホールが暴走崩壊！局所的なブラックホールが生成され全滅。",
          "next_scene": "scene_sp2",
          "is_gameover": True
        }
      ]
    },
    "scene_sp2": {
      "chapter_name": "番外編 SP2: ブラックホール超巨大エネルギー抽出（ペンローズ過程）",
      "description": "さらなる拡張のため、近隣の回転ブラックホール（カー・ブラックホール）の作用球（エルゴ球）から超巨大エネルギーを抽出する実験を開始！",
      "stage_id": "sp2",
      "image_file": "kenta_sp2_page1_manga.png",
      "options": [
        {
          "label": "エルゴ球へ物質を投入し、負のエネルギー部分を落下させて回転エネルギーを回収",
          "delta": {"fuel_tons": 100.0},
          "result_text": "【物理的帰結】ペンローズ過程が成功！理論限界に近い巨大エネルギーを抽出獲得。",
          "next_scene": "scene_sp3",
          "is_gameover": False
        },
        {
          "label": "超放射散乱（ブラックホール爆弾）構成を作り、電磁波を共振増幅させる",
          "delta": {"fuel_tons": 80.0, "hull_pct": -10.0},
          "result_text": "【物理的帰結】爆発的なエネルギーを得たが、プラズマの跳ね返りで外壁が損傷した。",
          "next_scene": "scene_sp3",
          "is_gameover": False
        },
        {
          "label": "安全な遠軌道からテザーケーブルを伸ばして重力勾配発電を行う",
          "delta": {"fuel_tons": 30.0},
          "result_text": "【物理的帰結】安全に持続可能な電力を確保した。",
          "next_scene": "scene_sp3",
          "is_gameover": False
        },
        {
          "label": "事象の地平線に限界まで接近し、制御噴射なしでエネルギー回収を試みる",
          "delta": {"hull_pct": -100.0},
          "result_text": "【物理的帰結】潮汐力により船体がスパゲッティ化！地平線の彼方へ消失した。",
          "next_scene": "scene_sp3",
          "is_gameover": True
        }
      ]
    },
    "scene_sp3": {
      "chapter_name": "番外編 SP3: 時空疲労と真空崩壊リスクの回避",
      "description": "ワームホール網の過剰増殖により、局所的な時空疲労と偽の真空の崩壊（真空崩壊）リスクが検知された。物理AIケンタの制御判断が問われる！",
      "stage_id": "sp3",
      "image_file": "kenta_sp3_page1_manga.png",
      "options": [
        {
          "label": "不要なワームホール口を統合・閉鎖し、エキゾチック物質の位相を緩やかに相殺する",
          "delta": {"delay_years": 0.2},
          "result_text": "【物理的帰結】安全に歪みが消滅。安定した長期運用が可能となった。",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        },
        {
          "label": "負のエネルギー密度（エキゾチック物質）を注入して喉を緊急冷却",
          "delta": {"fuel_tons": -10.0},
          "result_text": "【物理的帰結】燃料を消費して時空の揺らぎを強力に押さえ込んだ。",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        },
        {
          "label": "時空歪みを銀河外縁の真空領域へと緩やかに排熱・放射拡散させる",
          "delta": {"hull_pct": 0.0},
          "result_text": "【物理的帰結】宇宙論的スケールでの安全な熱・位相平衡が達成された。",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        },
        {
          "label": "警告メッセージを無視し、全ワームホールの送受信出力を一斉に限界突破させる",
          "delta": {"hull_pct": -100.0},
          "result_text": "【物理的帰結】真の真空への相転移が勃発！光速で拡大する気泡がすべてを消滅させた。",
          "next_scene": "scene_epilogue",
          "is_gameover": True
        }
      ]
    },
    "scene_epilogue": {
      "chapter_name": "最終話「時超える絆」: 星間ハビタブルゾーン同盟の発足",
      "description": "地球とプロキシマbが時空を超えて直結された。ケンタは銀河すべての星間ハビタブルゾーンを守護する存在となり、人類の新しい未来が始まった！",
      "stage_id": "epilogue",
      "image_file": "kenta_epilogue_page1_manga.png",
      "options": []
    }
  }
}

# ------------------------------------------------------------------------------
# 3. ゲームエンジン本体
# ------------------------------------------------------------------------------
class KentaGameEngine:
    def __init__(self):
        self.db = EMBEDDED_DB
        self._load_external_json_if_exists()
        self.state = dict(self.db.get("initial_state", {}))
        self.current_scene_id = "scene_prologue"

    def _load_external_json_if_exists(self):
        """解凍されたフォルダ内に有効な .json があれば優先ロード"""
        json_matches = glob.glob("/content/**/*.json", recursive=True) + glob.glob("./*.json")
        for jpath in json_matches:
            if "kenta_game_db" in os.path.basename(jpath) or "kenta" in os.path.basename(jpath):
                try:
                    with open(jpath, "r", encoding="utf-8") as f:
                        data = json.load(f)
                        if "scenes" in data and "scene_prologue" in data["scenes"]:
                            self.db = data
                            print("📖 有効な外部シナリオJSONを読み込みました:", jpath)
                            break
                except Exception:
                    pass

    def render(self):
        clear_output(wait=True)
        scene = self.db.get("scenes", {}).get(self.current_scene_id)

        if not scene:
            print("🏁 【ミッション完了】星間ハビタブルゾーン同盟（IHZU）が創設されました！おめでとうございます！")
            return

        # 1. ステータス表示
        print("=" * 70)
        print("🚀 【時超えケンタ (CHRONO KENTA) - 本編＆番外編アドベンチャー】")
        print("=" * 70)
        print(f"  速度: {self.state.get('velocity_c', 0.2):.2f} c | 装甲: {self.state.get('hull_pct', 100):.1f}% | 燃料: {self.state.get('fuel_tons', 100):.1f} t")
        print(f"  クローラー: {self.state.get('crawlers_count', 10)} 台 | 通信遅延: {self.state.get('delay_years', 4.2):.1f} 年")
        print("=" * 70)
        print()

        # 2. シーンテキスト
        print(f"📖 【{scene.get('chapter_name', scene.get('title', ''))}】")
        print(f"  {scene.get('description', '')}")
        print()

        # 3. 正史漫画コマ画像検索＆表示
        img_name = scene.get("image_file", "")
        img_location, is_url = find_image_file(img_name)

        print("📸 --- [正史ストーリーコマ] ---")
        if img_location:
            try:
                if is_url:
                    display(Image(url=img_location, width=650))
                else:
                    display(Image(filename=img_location, width=650))
            except Exception as e:
                print(f"🖼️ [正史コマ枠]: {img_name}")
        else:
            print(f"🖼️ [正史コマ枠]: {img_name}")

        print()

        # 4. ボタンUI生成（順序はランダムシャッフル）
        options = scene.get("options", [])
        if options:
            print("💡 【物理的決断を選んでください（ボタン順序ランダム）】:")
            shuffled_options = list(options)
            random.shuffle(shuffled_options)

            for idx, opt in enumerate(shuffled_options, 1):
                btn = widgets.Button(
                    description=f"[{idx}] {opt['label'][:42]}...",
                    button_style='primary',
                    layout=widgets.Layout(width='95%', height='42px', margin='4px')
                )

                def make_handler(option_data):
                    return lambda b: self.choose(option_data)

                btn.on_click(make_handler(opt))
                display(btn)
        else:
            print("🎉 【全シナリオ完了】同盟の未来が確立されました！")

    def choose(self, option):
        clear_output(wait=True)
        print("=" * 70)
        print("📢 選択の物理的結果")
        print("=" * 70)

        res_text = option.get('result_text', '')
        print(res_text)
        print()

        for k, v in option.get("delta", {}).items():
            if k in self.state:
                self.state[k] = max(0.0, self.state[k] + v)

        if option.get("is_gameover", False):
            print("💀 通信絶望 - 探査不能となりました。")
            retry_btn = widgets.Button(description="🔄 チェックポイントからやり直す", button_style='danger')
            retry_btn.on_click(lambda b: self.reset())
            display(retry_btn)
        else:
            self.current_scene_id = option.get("next_scene")
            next_btn = widgets.Button(description="▶ 次のステージ（または番外編）へ進む", button_style='success')
            next_btn.on_click(lambda b: self.render())
            display(next_btn)

    def reset(self):
        self.state = dict(self.db.get("initial_state", {}))
        self.current_scene_id = "scene_prologue"
        self.render()

# ゲーム開始
engine = KentaGameEngine()
engine.render()